# Hardware-robustness addendum (v3): DEV-BR under gain, clipping, ADC-range and sample-rate changes

Re-runs only the fast firmware stage on CHB-MIT (FP1–FP2) under seven front-end conditions and evaluates DEV-BR on each.
The key test applies the parameters chosen at ×1000 unchanged to every other condition.
Run time is about 20–40 minutes, all on Kaggle.

### How to run
1. In the editor's right-hand panel (**Session options**), switch **Internet → On**. Accelerator: **None**.
2. Click **Save Version → Save & Run All (Commit)**.
3. When it finishes, open **Output** and download **`vera_hw_results.zip`** (a few MB).

In [ ]:
import os, sys, socket, subprocess, time, json, shutil
from pathlib import Path
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATA = os.environ.get("VERA_DATA", f"{WORK}/data/chbmit")
OUT  = f"{WORK}/results_hw"
JOBS = os.cpu_count() or 2
def sh(cmd):
    print("$", cmd, flush=True); r = subprocess.run(cmd, shell=True, text=True)
    if r.returncode: raise RuntimeError(f"failed ({r.returncode}): {cmd}")
if not os.environ.get("VERA_DATA"):
    try: socket.create_connection(("physionet-open.s3.amazonaws.com", 443), timeout=8).close(); print("internet OK")
    except OSError: raise SystemExit("No internet: Session options -> Internet -> ON, then Save & Run All again.")
for pkg in ["numba", "pandas", "scipy", "requests"]:
    try: __import__(pkg)
    except ImportError: sh(f"{sys.executable} -m pip install -q {pkg}")
Path(f"{WORK}/evaluation").mkdir(exist_ok=True)

In [ ]:
%%writefile {WORK}/evaluation/chbmit.py
"""
CHB-MIT loading + Vera front-end simulation.

Data files are produced by `fetch_chbmit.js` (run in a browser on the PhysioNet
S3 mirror): for every EDF it stores only the FP1-F7 / FP2-F8 channels as raw
little-endian int16 digital values (`<rec>.ch.bin`, interleaved [n, n_ch]) plus
the EDF calibration header (`<rec>.ch.json`).  Nothing else is altered.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass
from pathlib import Path

import numpy as np
from scipy.signal import resample_poly

# ── Vera analog front-end model (see README / PIN_CONNECTIONS) ───────────────
FRONTEND_GAIN = 1000.0      # EXG BioAmp Pill gain
ADC_VREF = 3.3              # ESP32 ADC full scale (V), 12-bit
ADC_BITS = 12
ADC_MID = 2 ** (ADC_BITS - 1)   # single-supply mid-rail bias -> 2048
DEVICE_FS = 200             # firmware samples at ~200 Hz (delay(5))


@dataclass
class Record:
    case: str
    name: str
    fs: float
    uv: dict          # channel label -> float32 µV array at native fs
    seizures: list    # [(onset_s, offset_s), ...]

    @property
    def duration_s(self) -> float:
        return len(next(iter(self.uv.values()))) / self.fs


def parse_summary(path: Path) -> dict:
    """Return {edf_name: [(onset, offset), ...]} for every file in a summary."""
    txt = Path(path).read_text(errors="replace")
    out = {}
    for block in re.split(r"File Name:\s*", txt)[1:]:
        name = block.split()[0].strip()
        starts = [int(x) for x in re.findall(r"Seizure\s*\d*\s*Start Time:\s*(\d+)", block)]
        ends = [int(x) for x in re.findall(r"Seizure\s*\d*\s*End Time:\s*(\d+)", block)]
        n = re.search(r"Number of Seizures in File:\s*(\d+)", block)
        n = int(n.group(1)) if n else len(starts)
        if len(starts) != n or len(ends) != n:
            raise ValueError(f"{path}: seizure count mismatch for {name}")
        out[name] = list(zip(starts, ends))
    return out


def load_record(data_dir: Path, case: str, name: str, seizures) -> Record:
    meta = json.loads((data_dir / f"{name}.ch.json").read_text())
    raw = np.fromfile(data_dir / f"{name}.ch.bin", dtype="<i2")
    nch = len(meta["labels"])
    raw = raw.reshape(-1, nch).astype(np.float64)
    uv = {}
    for i, lab in enumerate(meta["labels"]):
        pmin, pmax = meta["phys_min"][i], meta["phys_max"][i]
        dmin, dmax = meta["dig_min"][i], meta["dig_max"][i]
        gain = (pmax - pmin) / (dmax - dmin)
        x = (raw[:, i] - dmin) * gain + pmin
        unit = meta["dim"][i].lower()
        if unit in ("mv",):
            x = x * 1e3
        elif unit in ("v",):
            x = x * 1e6
        uv[lab.upper()] = x.astype(np.float32)
    return Record(case, name, float(meta["fs"]), uv, list(seizures))


FRONTEND_BAND = (0.5, 40.0)   # EXG BioAmp Pill pass-band per the Vera circuit diagram


def frontend_filter(uv: np.ndarray, fs: float, band=FRONTEND_BAND) -> np.ndarray:
    """Causal model of the analog pass-band: 1st-order HP + 2nd-order Butterworth LP."""
    if band is None:
        return uv
    from scipy.signal import butter, lfilter
    lo, hi = band
    b, a = butter(1, lo, btype="highpass", fs=fs)
    y = lfilter(b, a, uv)
    b, a = butter(2, hi, btype="lowpass", fs=fs)
    return lfilter(b, a, y)


def to_device_adc(uv: np.ndarray, fs_in: float, gain: float = FRONTEND_GAIN,
                  band=FRONTEND_BAND, fs_out: float = DEVICE_FS, vref: float = ADC_VREF,
                  clip_hi_v: float | None = None) -> np.ndarray:
    """µV at fs_in  ->  integer ESP32 ADC codes at 200 Hz, as Vera would see them.

    analog pass-band (optional) -> resample (polyphase, anti-aliased) -> ×gain
    -> +mid-rail -> quantise -> clip.
    """
    from fractions import Fraction
    x = frontend_filter(uv.astype(np.float64), fs_in, band)
    fr = Fraction(fs_out / fs_in).limit_denominator(1000)
    x = resample_poly(x, fr.numerator, fr.denominator)
    volts = 1.65 + x * 1e-6 * gain                  # BioAmp output around mid-supply (3.3 V / 2)
    if clip_hi_v is not None:                       # e.g. LM358 output/common-mode ceiling
        volts = np.minimum(volts, clip_hi_v)
    codes = np.round(volts / (vref / 2 ** ADC_BITS))
    return np.clip(codes, 0, 2 ** ADC_BITS - 1).astype(np.int32)


def discover(data_root: Path):
    """Yield (case, edf_name, seizures, data_dir) for every extracted record."""
    data_root = Path(data_root)
    for summ in sorted(data_root.glob("chb*-summary.txt")):
        case = summ.name.split("-")[0]
        table = parse_summary(summ)
        for edf, sz in table.items():
            name = edf.replace(".edf", "")
            if (data_root / f"{name}.ch.json").exists():
                yield case, name, sz, data_root


In [ ]:
%%writefile {WORK}/evaluation/detectors.py
"""
Offline replay of Vera's two detectors.

1. `run_python_detector` drives the *unmodified* seizure_monitor.py through its
   own `serial_reader()` loop: a fake serial port emits the firmware's
   "eeg baseline diff state thr" lines and a simulated clock advances 5 ms per
   line, so warm-up / re-adaptation timing is exactly what it would be live.
   Only I/O side-effects (alerts, JSONL log, plotting backend) are stubbed.

2. `run_firmware_detector` is a line-by-line port of loop() in
   seizure_monitor_esp32_firmware.ino (EMA baseline, |x-baseline| diff, Welford
   EWMA, adaptive μ+Kσ thresholds, confirm counters, post-ictal hold), JIT
   compiled with numba.  C integer casts/truncation are reproduced.

Both return a per-tick state trace: (times_s, states) with states coded
0=NORMAL 1=BROWNOUT 2=BLACKOUT 3=POST-ICTAL.
"""
from __future__ import annotations

import importlib.util
import logging
import sys
import types
from pathlib import Path

import numpy as np
from numba import njit

STATE_CODE = {"NORMAL": 0, "BROWNOUT": 1, "BLACKOUT": 2, "POST-ICTAL": 3}
FS = 200.0
DT = 1.0 / FS


# ═════════════════════════════════════════════════════════════════════════════
#  Python analyzer (seizure_monitor.py) — unmodified module, replayed
# ═════════════════════════════════════════════════════════════════════════════
class _SimClock:
    def __init__(self):
        self.t = 1_000_000.0

    def time(self):
        return self.t

    def sleep(self, _):
        pass

    def __getattr__(self, item):            # monotonic, perf_counter, …
        import time as _t
        return getattr(_t, item)


def _ensure_serial_module():
    """seizure_monitor.py imports pyserial; replay never opens a port, so a stub suffices."""
    try:
        import serial  # noqa: F401
    except ImportError:
        stub = types.ModuleType("serial")

        class SerialException(Exception):
            pass

        class Serial:  # pragma: no cover - never instantiated during replay
            def __init__(self, *a, **k):
                raise SerialException("pyserial stub: no real ports in offline replay")

        stub.Serial, stub.SerialException = Serial, SerialException
        sys.modules["serial"] = stub


def load_monitor_module(path: Path):
    _ensure_serial_module()
    import matplotlib
    matplotlib.use = lambda *a, **k: None   # TkAgg not needed headless
    matplotlib.rcParams["backend"] = "Agg"
    spec = importlib.util.spec_from_file_location("seizure_monitor", str(path))
    mod = importlib.util.module_from_spec(spec)
    sys.modules["seizure_monitor"] = mod
    spec.loader.exec_module(mod)
    logging.getLogger(mod.__name__).setLevel(logging.ERROR)
    logging.getLogger().setLevel(logging.ERROR)
    mod.log_event = lambda *a, **k: None
    mod.send_alert = lambda *a, **k: None
    return mod


class _FakeSerial:
    """Emits firmware-format serial lines and advances the simulated clock."""

    def __init__(self, mod, clock, adc, fw_baseline, fw_diff, fw_state, fw_thr):
        self.mod, self.clock = mod, clock
        self.adc, self.base, self.diff = adc, fw_baseline, fw_diff
        self.state, self.thr = fw_state, fw_thr
        self.i, self.n = 0, len(adc)

    def readline(self):
        if self.i >= self.n:
            self.mod.SHARED.running = False
            return b""
        i = self.i
        self.i += 1
        self.clock.t += DT
        return (f"{self.adc[i]} {self.base[i]:.2f} {self.diff[i]} "
                f"{self.state[i]} {self.thr[i]}\n").encode()


def run_python_detector(mod, adc, fw_trace, cfg_overrides=None):
    """Replay one recording through seizure_monitor.serial_reader()."""
    from collections import deque

    clock = _SimClock()
    mod.time = clock                                   # module-level `time`
    mod.CFG = mod.Config(**(cfg_overrides or {}))
    mod.ADAPT = mod.AdaptiveThresholds()               # start = sim time 0
    mod.SHARED = mod.SharedState()
    mod._prev_seizure_state = "NORMAL"
    mod._confirm_buf = deque(maxlen=mod.CFG.confirm_ticks)
    # `update_seizure_state` reads _confirm_buf as a module global → rebinding works

    times, states, feats, thr = [], [], [], []
    orig = getattr(mod.update_seizure_state, "__wrapped__", mod.update_seizure_state)
    orig_an = getattr(mod.analyse, "__wrapped__", mod.analyse)

    def traced(raw_class, alpha_r, beta_r, ll, dar, hfe):
        s = orig(raw_class, alpha_r, beta_r, ll, dar, hfe)
        times.append(clock.t - 1_000_000.0)
        states.append(STATE_CODE[s])
        if len(states) % 10 == 1:                      # thresholds at ~1 Hz
            A = mod.ADAPT
            thr.append((clock.t - 1_000_000.0, A.ll_brownout, A.dar_brownout, A.hfe_brownout,
                        A.ll_blackout, A.dar_blackout, A.hfe_blackout))
        return s

    def traced_an(samples):                            # every tick, incl. artifacts
        r = orig_an(samples)
        feats.append((clock.t - 1_000_000.0, r[4], r[5], r[6]))
        return r

    traced.__wrapped__ = orig; traced_an.__wrapped__ = orig_an
    mod.update_seizure_state = traced; mod.analyse = traced_an

    base, diff, st, thr_fw = fw_trace
    ser = _FakeSerial(mod, clock, adc, base, diff, st, thr_fw)
    try:
        mod.serial_reader(ser)
    finally:
        mod.update_seizure_state = orig; mod.analyse = orig_an
    return (np.asarray(times), np.asarray(states, dtype=np.int8), mod.SHARED.artifact_count,
            np.asarray(feats, dtype=np.float32).reshape(-1, 4),
            np.asarray(thr, dtype=np.float32).reshape(-1, 7))


# ═════════════════════════════════════════════════════════════════════════════
#  ESP32 firmware detector — faithful port of loop()
# ═════════════════════════════════════════════════════════════════════════════
FW = dict(
    EMA_ALPHA=0.01, ADAPT_WARMUP_MS=90000, ADAPT_INTERVAL_MS=30000,
    Z_B=2.5, Z_K=4.0, B_FLOOR=60, K_FLOOR=120, B_CEIL=1200, K_CEIL=2400,
    WF_FORGET=0.995, CONFIRM_B=20, CONFIRM_K=15, POSTICTAL_HOLD=400,
    SAMPLE_MS=5,
)


@njit(cache=True)
def _constrain(x, lo, hi):
    return lo if x < lo else (hi if x > hi else x)


@njit(cache=True)
def _fw_loop(adc, ema_a, warm_ms, adapt_ms, zb, zk, bf, kf, bc, kc, lam,
             cb, ck, hold, sample_ms):
    n = adc.shape[0]
    # setup(): baseline = integer mean of first 200 samples (long sum / 200)
    ncal = 200 if n > 200 else n
    s = 0
    for i in range(ncal):
        s += adc[i]
    baseline = np.float32(s // ncal)
    # Welford seed
    wn = 0.0; wm = 0.0; wM2 = 0.0
    for _ in range(20):
        wn = lam * wn + 1.0
        d = 10.0 - wm
        wm += d / wn
        wM2 = lam * wM2 + d * (10.0 - wm)
    thrB = 80; thrK = 160
    state = 0; cB = 0; cK = 0; pcount = 0
    start_ms = ncal * sample_ms
    last_adapt = start_ms
    out_base = np.empty(n - ncal, np.float32)
    out_diff = np.empty(n - ncal, np.int32)
    out_state = np.empty(n - ncal, np.int8)
    out_thr = np.empty(n - ncal, np.int32)
    one_m = np.float32(1.0) - np.float32(ema_a)
    a32 = np.float32(ema_a)
    for j in range(n - ncal):
        i = j + ncal
        now = i * sample_ms
        x = adc[i]
        baseline = one_m * baseline + a32 * np.float32(x)
        diff = abs(x - int(baseline))
        if state == 0:
            xf = np.float64(np.float32(diff))
            wn = lam * wn + 1.0
            d = xf - wm
            wm += d / wn
            wM2 = lam * wM2 + d * (xf - wm)
        if (now - start_ms) >= warm_ms and (now - last_adapt) >= adapt_ms:
            mu = np.float32(wm)
            sig = np.float32(0.0) if wn < 2.0 else np.float32(np.sqrt(wM2 / wn))
            if sig >= 1.0:
                nb = _constrain(int(mu + np.float32(zb) * sig), bf, bc)
                nk = _constrain(int(mu + np.float32(zk) * sig), kf, kc)
                if nk <= nb:
                    nk = nb + 40
                thrB = nb; thrK = nk
            last_adapt = now
        if state == 0:
            if diff >= thrK:
                cK += 1
                if cK >= ck:
                    state = 2; cK = 0; cB = 0
            elif diff >= thrB:
                cK = 0
                cB += 1
                if cB >= cb:
                    state = 1; cB = 0
            else:
                if cB > 0: cB -= 1
                if cK > 0: cK -= 1
        elif state == 1:
            if diff >= thrK:
                cK += 1
                if cK >= ck:
                    state = 2; cK = 0
            elif diff < thrB:
                cB += 1
                if cB >= cb:
                    state = 0; cB = 0
            else:
                cB = 0; cK = 0
        elif state == 2:
            if diff < thrB:
                state = 3; pcount = hold
        else:
            pcount -= 1
            if pcount <= 0 and diff < thrB:
                state = 0
        out_base[j] = baseline; out_diff[j] = diff
        out_state[j] = state; out_thr[j] = thrB
    return ncal, out_base, out_diff, out_state, out_thr


def run_firmware_detector(adc, **over):
    p = dict(FW, **over)
    ncal, base, diff, st, thr = _fw_loop(
        np.ascontiguousarray(adc, dtype=np.int32), p["EMA_ALPHA"],
        p["ADAPT_WARMUP_MS"], p["ADAPT_INTERVAL_MS"], p["Z_B"], p["Z_K"],
        p["B_FLOOR"], p["K_FLOOR"], p["B_CEIL"], p["K_CEIL"], p["WF_FORGET"],
        p["CONFIRM_B"], p["CONFIRM_K"], p["POSTICTAL_HOLD"], p["SAMPLE_MS"])
    t = (np.arange(len(st)) + ncal) * DT
    return ncal, t, st, (base, diff, st, thr)


In [ ]:
%%writefile {WORK}/evaluation/scoring.py
"""
Event-based scoring (SzCORE-style any-overlap with tolerances).

An *alarm* is the moment the detector enters an alarm state from a non-alarm
state.  Two alarm definitions are reported:
  • "seizure alarm"  : entry into BLACKOUT                (the urgent alert)
  • "any alarm"      : entry into BROWNOUT or BLACKOUT     (warning or urgent)
Alarms closer than MERGE_S to the previous alarm are merged (one alert).

A seizure is detected if an alarm falls in [onset − PRE_S, offset + POST_S].
Latency = first such alarm − onset (negative = before annotated onset).
Every alarm that falls in no seizure's window is a false alarm.
"""
from __future__ import annotations

import numpy as np

PRE_S = 30.0
POST_S = 60.0
MERGE_S = 90.0


def alarm_times(t, states, kind="seizure"):
    t = np.asarray(t); s = np.asarray(states)
    alarm = (s == 2) if kind == "seizure" else ((s == 1) | (s == 2))
    prev = np.concatenate([[False], alarm[:-1]])
    onsets = t[alarm & ~prev]
    merged = []
    for a in onsets:
        if not merged or a - merged[-1] >= MERGE_S:
            merged.append(float(a))
    return merged


def score_record(alarms, seizures, duration_s):
    det, lat = [], []
    used = set()
    for (on, off) in seizures:
        hits = [a for a in alarms if on - PRE_S <= a <= off + POST_S]
        det.append(bool(hits))
        lat.append(min(hits) - on if hits else np.nan)
        used.update(hits)
    fa = [a for a in alarms if a not in used]
    return dict(n_sz=len(seizures), detected=det, latency=lat,
                n_fa=len(fa), fa_times=fa, hours=duration_s / 3600.0)


def summarise(rows):
    """rows: list of dicts from score_record (+ 'case').  Pooled + per-case."""
    n_sz = sum(r["n_sz"] for r in rows)
    n_det = sum(sum(r["detected"]) for r in rows)
    hours = sum(r["hours"] for r in rows)
    n_fa = sum(r["n_fa"] for r in rows)
    lats = [l for r in rows for l in r["latency"] if not np.isnan(l)]
    per_case = {}
    for r in rows:
        c = per_case.setdefault(r["case"], dict(n_sz=0, n_det=0, n_fa=0, hours=0.0, lat=[]))
        c["n_sz"] += r["n_sz"]; c["n_det"] += sum(r["detected"])
        c["n_fa"] += r["n_fa"]; c["hours"] += r["hours"]
        c["lat"] += [l for l in r["latency"] if not np.isnan(l)]
    case_sens = [c["n_det"] / c["n_sz"] for c in per_case.values() if c["n_sz"]]
    return dict(
        seizures=n_sz, detected=n_det,
        sensitivity=n_det / n_sz if n_sz else np.nan,
        mean_case_sensitivity=float(np.mean(case_sens)) if case_sens else np.nan,
        hours=hours, false_alarms=n_fa, fa_per_hour=n_fa / hours if hours else np.nan,
        fa_per_24h=24 * n_fa / hours if hours else np.nan,
        latency_median=float(np.median(lats)) if lats else np.nan,
        latency_iqr=(float(np.percentile(lats, 25)), float(np.percentile(lats, 75))) if lats else (np.nan, np.nan),
        per_case=per_case,
    )


In [ ]:
%%writefile {WORK}/evaluation/fetch_chbmit.py
"""
Fetch CHB-MIT recordings and keep only the channels Vera can see.

For every EDF: download (PhysioNet AWS open-data mirror, or read from a local /
attached copy), extract FP1-F7 and FP2-F8 as raw int16 + calibration header
(`<rec>.ch.bin` / `<rec>.ch.json`), then delete the EDF.  If a recording uses a
referential montage (e.g. some chb12 files), the bipolar channel is derived as
FP1-ref − F7-ref and flagged `derived` in the json.

Selection (default): every recording that contains a seizure (141 files,
198 seizures) + the first seizure-free recording of every patient
(+ chb01_01 and chb01_02) for additional false-alarm hours.

usage: python fetch_chbmit.py --out data/chbmit [--local /kaggle/input/...] [--workers 6]
"""
from __future__ import annotations

import argparse
import json
import os
import re
import sys
import time
import urllib.parse
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np

S3 = "https://physionet-open.s3.amazonaws.com/chbmit/1.0.0/"
WANT = {"FP1-F7": ("FP1", "F7"), "FP2-F8": ("FP2", "F8")}
CASES = [f"chb{i:02d}" for i in range(1, 25)]


def _get(url, tries=4):
    import requests
    for k in range(tries):
        try:
            r = requests.get(url, timeout=120)
            r.raise_for_status()
            return r.content
        except Exception:
            if k == tries - 1:
                raise
            time.sleep(2 * (k + 1))


def read_edf_channels(buf: bytes):
    """Minimal EDF reader → (meta, {label: int16 array})."""
    h = lambda o, l: buf[o:o + l].decode("ascii", "replace").strip()
    nrec, dur, ns = int(h(236, 8)), float(h(244, 8)), int(h(252, 4))
    o = 256
    def field(w):
        nonlocal o
        v = [h(o + i * w, w) for i in range(ns)]; o += ns * w; return v
    labels = field(16); field(80); dim = field(8)
    pmin = list(map(float, field(8))); pmax = list(map(float, field(8)))
    dmin = list(map(float, field(8))); dmax = list(map(float, field(8)))
    field(80); nsamp = list(map(int, field(8))); field(32)
    hdr = 256 + ns * 256
    rec_len = sum(nsamp)
    n_real = (len(buf) - hdr) // (rec_len * 2)
    nrec = min(nrec if nrec > 0 else n_real, n_real)
    data = np.frombuffer(buf, dtype="<i2", count=nrec * rec_len, offset=hdr).reshape(nrec, rec_len)
    offs = np.concatenate([[0], np.cumsum(nsamp)[:-1]])
    def chan(i):
        return data[:, offs[i]:offs[i] + nsamp[i]].reshape(-1)
    return dict(labels=labels, dim=dim, pmin=pmin, pmax=pmax, dmin=dmin, dmax=dmax,
                nsamp=nsamp, dur=dur), chan


def _bipolar_path(labels_upper, a, b):
    """Shortest chain of bipolar channels from electrode a to b → [(index, ±1), ...]."""
    from collections import deque
    adj = {}
    for k, l in enumerate(labels_upper):
        parts = l.split("-")
        if len(parts) != 2 or not parts[0] or not parts[1]:
            continue
        x, y = parts
        adj.setdefault(x, []).append((y, k, +1))     # x - y : stepping x→y adds +(x−y)
        adj.setdefault(y, []).append((x, k, -1))     # stepping y→x adds −(x−y)
    prev = {a: None}
    q = deque([a])
    while q:
        u = q.popleft()
        if u == b:
            break
        for v, k, s in adj.get(u, []):
            if v not in prev:
                prev[v] = (u, k, s); q.append(v)
    if b not in prev:
        return None
    path, node = [], b
    while prev[node] is not None:
        u, k, s = prev[node]; path.append((k, s)); node = u
    return path[::-1]


def extract(buf: bytes, source: str, out_dir: Path, name: str):
    H, chan = read_edf_channels(buf)
    # plain electrode names (e.g. chb12_28: "FP1", "F7", … against a common
    # reference) are treated as "<name>-REF" so referential montages work too
    up = [l.upper() if ("-" in l or not l.strip()) else l.upper().strip() + "-REF" for l in H["labels"]]
    cols, meta = [], dict(source=source, labels=[], dim=[], phys_min=[], phys_max=[],
                          dig_min=[], dig_max=[], derived=[], missing=[], all_labels=H["labels"])
    for want, (a, b) in WANT.items():
        if want in up:
            i = up.index(want)
            cols.append(chan(i).astype(np.int16))
            meta["labels"].append(want); meta["dim"].append(H["dim"][i])
            meta["phys_min"].append(H["pmin"][i]); meta["phys_max"].append(H["pmax"][i])
            meta["dig_min"].append(H["dmin"][i]); meta["dig_max"].append(H["dmax"][i])
            meta["derived"].append(False)
            fs = H["nsamp"][i] / H["dur"]
            continue
        # referential fallback: "FP1-<ref>" and "F7-<ref>" with the same ref
        ia = [k for k, l in enumerate(up) if l.split("-")[0] == a and "-" in l]
        ib = [k for k, l in enumerate(up) if l.split("-")[0] == b and "-" in l]
        pair = next(((i, j) for i in ia for j in ib if up[i].split("-", 1)[1] == up[j].split("-", 1)[1]), None)
        if pair is None:
            meta["missing"].append(want); continue
        i, j = pair
        # convert both to physical units, subtract, re-quantise on channel i's scale
        g = lambda k: (H["pmax"][k] - H["pmin"][k]) / (H["dmax"][k] - H["dmin"][k])
        phys = lambda k: (chan(k).astype(np.float64) - H["dmin"][k]) * g(k) + H["pmin"][k]
        d = phys(i) - phys(j)
        q = np.clip(np.round((d - H["pmin"][i]) / g(i) + H["dmin"][i]), -32768, 32767).astype(np.int16)
        cols.append(q)
        meta["labels"].append(want); meta["dim"].append(H["dim"][i])
        meta["phys_min"].append(H["pmin"][i]); meta["phys_max"].append(H["pmax"][i])
        meta["dig_min"].append(H["dmin"][i]); meta["dig_max"].append(H["dmax"][i])
        meta["derived"].append(f"{H['labels'][i]} - {H['labels'][j]}")
        fs = H["nsamp"][i] / H["dur"]
    # Vera's own montage: FP1 − FP2, reconstructed exactly as a signed sum of a
    # chain of bipolar channels (e.g. FP1-F7, F7-T7, T7-FT9, FT9-FT10, FT10-T8, F8-T8, FP2-F8)
    chain = _bipolar_path(up, "FP1", "FP2")
    if chain:
        g = lambda k: (H["pmax"][k] - H["pmin"][k]) / (H["dmax"][k] - H["dmin"][k])
        phys = lambda k: (chan(k).astype(np.float64) - H["dmin"][k]) * g(k) + H["pmin"][k]
        n0 = min(len(chan(k)) for k, _ in chain)
        d = sum(sgn * phys(k)[:n0] for k, sgn in chain)
        # store at 0.2 µV/LSB (±6553.4 µV) — finer than CHB-MIT's native resolution
        q = np.clip(np.round(d / 0.2), -32767, 32767).astype(np.int16)
        meta["fp1fp2_clip_frac"] = float(np.mean(np.abs(d / 0.2) > 32767))
        cols.append(q)
        meta["labels"].append("FP1-FP2"); meta["dim"].append("uV")
        meta["phys_min"].append(-6553.4); meta["phys_max"].append(6553.4)
        meta["dig_min"].append(-32767); meta["dig_max"].append(32767)
        meta["derived"].append(" + ".join(("" if s > 0 else "−") + H["labels"][k] for k, s in chain))
        fs = H["nsamp"][chain[0][0]] / H["dur"]
    else:
        meta["missing"].append("FP1-FP2")
    if not cols:
        return dict(name=name, error="no usable frontal channels", labels=H["labels"])
    n = min(len(c) for c in cols)
    arr = np.stack([c[:n] for c in cols], 1)
    meta.update(fs=fs, n=int(n))
    arr.astype("<i2").tofile(out_dir / f"{name}.ch.bin")
    (out_dir / f"{name}.ch.json").write_text(json.dumps(meta))
    return dict(name=name, n=int(n), fs=fs, labels=meta["labels"], derived=meta["derived"])


def select_files(summaries: dict) -> list:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
    from chbmit import parse_summary
    files = []
    for case in CASES:
        p = summaries[case]
        table = parse_summary(p)
        sz = [f for f, s in table.items() if s]
        free = [f for f, s in table.items() if not s]
        extra = free[:2] if case == "chb01" else free[:1]
        files += [f"{case}/{f}" for f in sz + extra]
    return files


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    ap.add_argument("--local", default=None, help="folder holding chbXX/ EDFs (e.g. an attached Kaggle dataset)")
    ap.add_argument("--workers", type=int, default=6)
    ap.add_argument("--all", action="store_true", help="every recording (≈42 GB download)")
    a = ap.parse_args()
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)

    local = None
    if a.local:
        hits = list(Path(a.local).rglob("chb01-summary.txt"))
        local = hits[0].parent.parent if hits else None
        print("local copy:", local)

    summaries = {}
    for c in CASES:
        dst = out / f"{c}-summary.txt"
        if not dst.exists():
            src = (local / c / f"{c}-summary.txt") if local else None
            dst.write_bytes(src.read_bytes() if src and src.exists() else _get(S3 + f"{c}/{c}-summary.txt"))
        summaries[c] = dst

    if a.all:
        from chbmit import parse_summary
        files = [f"{c}/{f}" for c in CASES for f in parse_summary(summaries[c])]
    else:
        files = select_files(summaries)
    todo = [f for f in files if not (out / (Path(f).stem + ".ch.json")).exists()]
    print(f"{len(files)} recordings selected, {len(todo)} to fetch", flush=True)

    def job(f):
        name = Path(f).stem
        lp = (local / f) if local else None
        if lp is not None and lp.exists():
            buf = lp.read_bytes()
        else:
            buf = _get(S3 + urllib.parse.quote(f))
        return extract(buf, "chbmit/1.0.0/" + f, out, name)

    log = []
    t0 = time.time()
    with ThreadPoolExecutor(a.workers) as ex:
        futs = {ex.submit(job, f): f for f in todo}
        for i, fu in enumerate(as_completed(futs), 1):
            f = futs[fu]
            try:
                r = fu.result()
            except Exception as e:                       # noqa: BLE001
                r = dict(name=f, error=repr(e))
            log.append(r)
            if i % 10 == 0 or "error" in r:
                print(f"[{i}/{len(todo)}] {f} {r.get('error', '')}  ({time.time()-t0:.0f}s)", flush=True)
    with open(out / "fetch_log.jsonl", "a") as fh:
        for r in log:
            fh.write(json.dumps(r) + "\n")
    errs = [r for r in log if "error" in r]
    print(f"done: {len(log) - len(errs)} ok, {len(errs)} errors")
    for r in errs:
        print("  ", r)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {WORK}/evaluation/personalize.py
"""
Scale-free ratio detectors and caregiver-calibrated personalization.

All detectors run on 1-Hz summaries of two signals Vera already computes:
  LL  – the analyzer's line-length (mean per second)
  DEV – the firmware's |x − EMA baseline| deviation (mean per second)
Each is divided by its own lagged running median (window [t−330, t−30] s), so
scores are dimensionless and independent of electrode impedance / gain.

  S_LL  = log R_LL          S_DEV = log R_DEV          S_FUSE = (S_LL + S_DEV)/2

Alarm: S ≥ θ for D consecutive seconds (alarms < 90 s apart merged).

Two ways of setting θ:
  • population   – θ = log r, (r, D) chosen by patient-wise 2-fold CV
  • personal (caregiver-calibrated) – the caregiver marks ONE seizure in the app.
    From that seizure alone the device takes p = the largest θ that would have
    raised an alarm (max over the seizure of the D-s running minimum of S),
    and sets θ = max(α·p, log f).  (α, f, D) are chosen by patient-wise 2-fold CV
    on *other* patients; the patient's own data contribute only the one
    calibration seizure, and every recording from that calibration file is
    excluded from evaluation.  The calibration seizure is the patient's first
    (chronological) seizure.

usage: python personalize.py --results results --channel FP1-FP2
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from scoring import MERGE_S, PRE_S, POST_S, score_record, summarise   # noqa: E402

BG_WIN, BG_LAG = 300, 30
DEAD = BG_WIN + BG_LAG


# ── 1-Hz scores ──────────────────────────────────────────────────────────────
def per_second(t, x, n):
    sec = np.clip(t.astype(np.int64), 0, n - 1)
    s = np.bincount(sec, weights=x.astype(np.float64), minlength=n)
    c = np.bincount(sec, minlength=n)
    y = np.where(c > 0, s / np.maximum(c, 1), np.nan)
    idx = np.where(np.isfinite(y), np.arange(n), 0)
    np.maximum.accumulate(idx, out=idx)
    return y[idx]


def log_ratio(y):
    import pandas as pd
    bg = pd.Series(y).rolling(BG_WIN, min_periods=BG_WIN // 2).median().shift(BG_LAG).to_numpy()
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.log(np.maximum(y, 1e-9) / np.maximum(bg, 1e-9))
    r[:DEAD] = np.nan
    return r


def scores_for(feat_path, duration_s):
    z = np.load(feat_path)
    n = int(np.floor(duration_s))
    ll = per_second(z["ft"], z["ll"], n)
    dev = per_second(z["fw_t10"], z["fw_diff10"], n)
    s_ll, s_dev = log_ratio(ll), log_ratio(dev)
    return {"LL": s_ll, "DEV": s_dev, "FUSE": 0.5 * (s_ll + s_dev)}


# ── alarms ───────────────────────────────────────────────────────────────────
def run_min(S, D):
    """running minimum over the last D seconds (NaN → -inf)."""
    x = np.nan_to_num(S, nan=-np.inf)
    if D <= 1:
        return x
    from numpy.lib.stride_tricks import sliding_window_view
    pad = np.concatenate([np.full(D - 1, -np.inf), x])
    return sliding_window_view(pad, D).min(axis=1)


def alarms(S, theta, D, _cache={}):
    m = run_min(S, D)
    above = m >= theta
    onsets = np.where(above & ~np.concatenate([[False], above[:-1]]))[0] + 1.0
    out = []
    for a in onsets:
        if not out or a - out[-1] >= MERGE_S:
            out.append(float(a))
    return out


# ── data assembly ────────────────────────────────────────────────────────────
def load_all(res: Path, channel: str):
    rows = [json.loads(l) for l in (res / "records.jsonl").read_text().splitlines() if l.strip()]
    rows = [r for r in rows if r["channel"] == channel and not r.get("missing")]
    rows.sort(key=lambda r: r["record"])
    for r in rows:
        r["S"] = scores_for(res / "features" / f"{r['record']}_{channel}.npz", r["duration_s"])
    return rows


def fold_of(case):
    return 1 if case == "chb21" else int(case[3:]) % 2


def evaluate(rows, sig, theta_fn, D):
    """theta_fn(row) -> theta (or None to skip row)."""
    sc = []
    for r in rows:
        th = theta_fn(r)
        if th is None:
            continue
        al = alarms(r["S"][sig], th, D)
        s = score_record(al, [tuple(x) for x in r["seizures"]], r["duration_s"])
        s["case"] = r["case"]; sc.append(s)
    return summarise(sc) if sc else None


# ── population thresholds (patient-wise 2-fold CV) ───────────────────────────
R_GRID = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0]
D_GRID = [2, 4, 6, 8, 10, 15, 20]


def population_cv(rows, sig, target):
    test_scored, chosen = [], []
    for f in (0, 1):
        tr = [r for r in rows if fold_of(r["case"]) != f]
        te = [r for r in rows if fold_of(r["case"]) == f]
        best = None
        for rr in R_GRID:
            for D in D_GRID:
                s = evaluate(tr, sig, lambda r, rr=rr: np.log(rr), D)
                if s["fa_per_hour"] <= target and (best is None or s["sensitivity"] > best[0]):
                    best = (s["sensitivity"], rr, D)
        if best is None:
            continue
        chosen.append((f, best[1], best[2]))
        for r in te:
            al = alarms(r["S"][sig], np.log(best[1]), best[2])
            s = score_record(al, [tuple(x) for x in r["seizures"]], r["duration_s"])
            s["case"] = r["case"]; test_scored.append(s)
    return summarise(test_scored), chosen


# ── personalization ──────────────────────────────────────────────────────────
A_GRID = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
F_GRID = [1.25, 1.5, 2.0, 2.5, 3.0]
DP_GRID = [2, 4, 6, 8, 10, 15]


def calibration_split(rows):
    """per patient: first seizure (chronological) → (calib_row, seizure); eval rows exclude that file."""
    by = {}
    for r in rows:
        by.setdefault(r["case"], []).append(r)
    plan = {}
    for c, rs in by.items():
        rs.sort(key=lambda r: r["record"])
        cal = next(((r, sz) for r in rs for sz in sorted(r["seizures"]) if sz[0] > DEAD + 5), None)
        if cal is None:
            continue
        ev = [r for r in rs if r["record"] != cal[0]["record"]]
        if not any(r["seizures"] for r in ev):
            continue
        plan[c] = (cal, ev)
    return plan


def peak_on(S, sz, D):
    on, off = int(sz[0]), int(sz[1])
    m = run_min(S, D)[on: off + int(POST_S) + 1]      # what the alarm logic could have seen
    m = m[np.isfinite(m)]
    return float(m.max()) if len(m) else -np.inf


def personal_eval(plan, cases, sig, a, f, D):
    sc = []
    for c in cases:
        (crow, csz), ev = plan[c]
        p = peak_on(crow["S"][sig], csz, D)
        th = max(a * p, np.log(f))
        for r in ev:
            al = alarms(r["S"][sig], th, D)
            s = score_record(al, [tuple(x) for x in r["seizures"]], r["duration_s"])
            s["case"] = r["case"]; sc.append(s)
    return summarise(sc)


def personal_cv(plan, sig, target):
    cases = sorted(plan)
    test, chosen = [], []
    for fo in (0, 1):
        tr = [c for c in cases if fold_of(c) != fo]
        te = [c for c in cases if fold_of(c) == fo]
        best = None
        for a in A_GRID:
            for f in F_GRID:
                for D in DP_GRID:
                    s = personal_eval(plan, tr, sig, a, f, D)
                    if s["fa_per_hour"] <= target and (best is None or s["sensitivity"] > best[0]):
                        best = (s["sensitivity"], a, f, D)
        if best is None:
            continue
        chosen.append((fo, best[1], best[2], best[3]))
        for c in te:
            (crow, csz), ev = plan[c]
            p = peak_on(crow["S"][sig], csz, best[3])
            th = max(best[1] * p, np.log(best[2]))
            for r in ev:
                al = alarms(r["S"][sig], th, best[3])
                s = score_record(al, [tuple(x) for x in r["seizures"]], r["duration_s"])
                s["case"] = r["case"]; test.append(s)
    return summarise(test), chosen


# ── wearer-baseline self-calibration (no seizure labels) ─────────────────────
# The device wears in for its first recordings (here: the patient's first two,
# chronologically — unlabeled, so any seizures in them are simply part of the
# data).  For each persistence D it picks the lowest θ whose alarm rate on that
# wear-in data stays ≤ q, then never adapts again.  (q, D) are chosen by
# patient-wise 2-fold CV; the wear-in recordings are excluded from evaluation.
TH_GRID = np.round(np.arange(0.05, 3.01, 0.05), 3)
Q_GRID = [0.25, 0.5, 1.0, 2.0]
DS_GRID = [4, 8, 10, 15, 20]
N_WEAR = 2


def wear_split(rows):
    by = {}
    for r in rows:
        by.setdefault(r["case"], []).append(r)
    plan = {}
    for c, rs in by.items():
        rs.sort(key=lambda r: r["record"])
        if len(rs) <= N_WEAR:
            continue
        cal, ev = rs[:N_WEAR], rs[N_WEAR:]
        if any(r["seizures"] for r in ev):
            plan[c] = (cal, ev)
    return plan


def self_theta(cal, sig, D, q):
    hours = sum(r["duration_s"] for r in cal) / 3600
    for th in TH_GRID:
        n = sum(len(alarms(r["S"][sig], th, D)) for r in cal)
        if n / hours <= q:
            return th
    return TH_GRID[-1]


def self_eval(plan, cases, sig, q, D):
    sc = []
    for c in cases:
        cal, ev = plan[c]
        th = self_theta(cal, sig, D, q)
        for r in ev:
            al = alarms(r["S"][sig], th, D)
            s = score_record(al, [tuple(x) for x in r["seizures"]], r["duration_s"])
            s["case"] = r["case"]; sc.append(s)
    return summarise(sc)


def self_cv(plan, sig, target):
    cases = sorted(plan); test, chosen = [], []
    for fo in (0, 1):
        tr = [c for c in cases if fold_of(c) != fo]; te = [c for c in cases if fold_of(c) == fo]
        best = None
        for q in Q_GRID:
            for D in DS_GRID:
                s = self_eval(plan, tr, sig, q, D)
                if s["fa_per_hour"] <= target and (best is None or s["sensitivity"] > best[0]):
                    best = (s["sensitivity"], q, D)
        if best is None:
            continue
        chosen.append((fo, best[1], best[2]))
        s = self_eval(plan, te, sig, best[1], best[2])
        for c, v in s["per_case"].items():
            pass
        for c in te:
            cal, ev = plan[c]
            th = self_theta(cal, sig, best[2], best[1])
            for r in ev:
                al = alarms(r["S"][sig], th, best[2])
                x = score_record(al, [tuple(y) for y in r["seizures"]], r["duration_s"])
                x["case"] = r["case"]; test.append(x)
    return summarise(test), chosen


def strip(s):
    return {k: v for k, v in s.items() if k != "per_case"}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", default="results")
    ap.add_argument("--channel", default="FP1-FP2")
    a = ap.parse_args()
    res = Path(a.results)
    rows = load_all(res, a.channel)
    out = dict(channel=a.channel, full={}, tap={}, wear={})
    md = [f"# Ratio detectors and personalization — {a.channel}\n"]

    # A. population ratio detectors on the full set
    md += ["## A. Population thresholds, all recordings (patient-wise 2-fold CV)",
           "| Signal | FA/h target | Params (fold0; fold1) | Detected | Sens. | FA/h | Latency (s) |", "|---|---|---|---|---|---|---|"]
    for tgt in (1.0, 0.5, 0.2):
        for sig in ("LL", "DEV", "FUSE"):
            s, ch = population_cv(rows, sig, tgt)
            out["full"][f"{sig}_{tgt}"] = dict(chosen=ch, **strip(s),
                per_case={c: dict(n_sz=v["n_sz"], n_det=v["n_det"], n_fa=v["n_fa"], hours=v["hours"]) for c, v in s["per_case"].items()})
            md.append(f"| {sig} | ≤{tgt} | " + "; ".join(f"r={r},D={d}" for _, r, d in ch) +
                      f" | {s['detected']}/{s['seizures']} | {100*s['sensitivity']:.1f}% | {s['fa_per_hour']:.2f} | {s['latency_median']:.1f} |")

    # B. caregiver-tap calibration vs population, same evaluation set
    plan = calibration_split(rows)
    ev_rows = [r for c in sorted(plan) for r in plan[c][1]]
    out["tap"]["eval"] = dict(patients=len(plan), seizures=sum(len(r["seizures"]) for r in ev_rows),
                              hours=sum(r["duration_s"] for r in ev_rows) / 3600)
    md += ["", f"## B. One caregiver-marked seizure (eval set: {out['tap']['eval']['patients']} patients, "
           f"{out['tap']['eval']['seizures']} seizures, {out['tap']['eval']['hours']:.1f} h)",
           "| Method | FA/h target | Detected | Sens. | FA/h |", "|---|---|---|---|---|"]
    for tgt in (1.0, 0.5):
        for sig in ("DEV", "FUSE"):
            s0, _ = population_cv(ev_rows, sig, tgt); s1, ch = personal_cv(plan, sig, tgt)
            out["tap"][f"{sig}_{tgt}"] = dict(population=strip(s0), personal=dict(chosen=ch, **strip(s1)))
            md.append(f"| population {sig} | ≤{tgt} | {s0['detected']}/{s0['seizures']} | {100*s0['sensitivity']:.1f}% | {s0['fa_per_hour']:.2f} |")
            md.append(f"| caregiver-tap {sig} | ≤{tgt} | {s1['detected']}/{s1['seizures']} | {100*s1['sensitivity']:.1f}% | {s1['fa_per_hour']:.2f} |")

    # C. wearer-baseline self-calibration vs population, same evaluation set
    wplan = wear_split(rows)
    wev = [r for c in sorted(wplan) for r in wplan[c][1]]
    out["wear"]["eval"] = dict(patients=len(wplan), seizures=sum(len(r["seizures"]) for r in wev),
                               hours=sum(r["duration_s"] for r in wev) / 3600,
                               wear_hours=sum(r["duration_s"] for c in wplan for r in wplan[c][0]) / 3600)
    md += ["", f"## C. Wearer-baseline self-calibration (eval set: {out['wear']['eval']['patients']} patients, "
           f"{out['wear']['eval']['seizures']} seizures, {out['wear']['eval']['hours']:.1f} h; first {N_WEAR} recordings = wear-in)",
           "| Method | FA/h target | Params | Detected | Sens. | FA/h | Latency (s) |", "|---|---|---|---|---|---|---|"]
    for tgt in (1.0, 0.5, 0.2):
        for sig in ("DEV", "FUSE"):
            s0, c0 = population_cv(wev, sig, tgt); s1, c1 = self_cv(wplan, sig, tgt)
            out["wear"][f"{sig}_{tgt}"] = dict(population=dict(chosen=c0, **strip(s0)), self=dict(chosen=c1, **strip(s1)))
            md.append(f"| population {sig} | ≤{tgt} | " + "; ".join(f"r={r},D={d}" for _, r, d in c0) +
                      f" | {s0['detected']}/{s0['seizures']} | {100*s0['sensitivity']:.1f}% | {s0['fa_per_hour']:.2f} | {s0['latency_median']:.1f} |")
            md.append(f"| self-calibrated {sig} | ≤{tgt} | " + "; ".join(f"q={q},D={d}" for _, q, d in c1) +
                      f" | {s1['detected']}/{s1['seizures']} | {100*s1['sensitivity']:.1f}% | {s1['fa_per_hour']:.2f} | {s1['latency_median']:.1f} |")

    (res / "tables").mkdir(exist_ok=True)
    (res / "tables" / f"personal_{a.channel}.md").write_text("\n".join(md) + "\n")
    (res / f"personal_{a.channel}.json").write_text(json.dumps(out, indent=2, default=float))
    print("\n".join(md))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {WORK}/evaluation/rigor.py
"""
Rigor analyses for the baseline-ratio (BR) detectors on the primary channel.

  1. Subject grouping: chb21 is the same person as chb01 → one subject ("S01").
  2. Leave-one-subject-out CV (LOSO, 23 folds) and the original 2-fold CV.
  3. Nested signal selection: inside each training fold the signal
     (LL / DEV / FUSE) is chosen together with (r, D), so the final choice of
     DEV is not made on held-out data.
  4. Pooled and subject-macro sensitivity, FA/h over total and inter-ictal time,
     subject-level bootstrap CIs, Wilcoxon tests on subjects.
  5. Empirical null: each detector's alarm train is circularly shifted by a
     random offset inside every recording (alarm count and spacing preserved)
     and re-scored, 1000 times.
  6. Ablations (LOSO): baseline lag, window length, median vs mean, and an
     un-normalized absolute threshold on the deviation.
  7. Cold-start accounting: seizures starting before the BR warm-up ends.

usage: python rigor.py --results results [--channel FP1-FP2] [--perm 1000]
writes results/rigor_<channel>.json
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from scoring import MERGE_S, PRE_S, POST_S, score_record                 # noqa: E402
import personalize as P                                                  # noqa: E402

R_GRID = [1.25, 1.5, 1.75, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0]
D_GRID = [2, 4, 6, 8, 10, 15, 20, 30]
TARGETS = [1.0, 0.5, 0.2]


def subject(case):
    return "chb01" if case == "chb21" else case


# ── score construction with configurable baseline ────────────────────────────
def baseline_score(y, win=300, lag=30, stat="median", log=True):
    import pandas as pd
    s = pd.Series(y)
    roll = s.rolling(win, min_periods=win // 2)
    bg = (roll.median() if stat == "median" else roll.mean()).shift(lag).to_numpy()
    if not log:
        return None
    with np.errstate(divide="ignore", invalid="ignore"):
        r = np.log(np.maximum(y, 1e-9) / np.maximum(bg, 1e-9))
    r[: win + lag] = np.nan
    return r


def series(res, rec, ch, n):
    z = np.load(res / "features" / f"{rec}_{ch}.npz")
    ll = P.per_second(z["ft"], z["ll"], n)
    dev = P.per_second(z["fw_t10"], z["fw_diff10"], n)
    return ll, dev


# ── fast alarm evaluation with cached running minima ─────────────────────────
class Cache:
    def __init__(self, rows, key):
        self.rows, self.key = rows, key
        self.m = {}

    def runmin(self, i, D):
        k = (i, D)
        if k not in self.m:
            self.m[k] = P.run_min(self.rows[i]["S"][self.key], D)
        return self.m[k]


def alarms_from_min(m, th):
    above = m >= th
    on = np.flatnonzero(above & ~np.concatenate([[False], above[:-1]])) + 1.0
    out = []
    for a in on:
        if not out or a - out[-1] >= MERGE_S:
            out.append(float(a))
    return out


def score_set(cache, idx, th, D):
    sc = []
    for i in idx:
        r = cache.rows[i]
        al = alarms_from_min(cache.runmin(i, D), th)
        s = score_record(al, [tuple(x) for x in r["seizures"]], r["duration_s"])
        s["subject"] = subject(r["case"]); s["alarms"] = al; s["i"] = i
        sc.append(s)
    return sc


def agg(sc):
    nsz = sum(s["n_sz"] for s in sc); ndet = sum(sum(s["detected"]) for s in sc)
    h = sum(s["hours"] for s in sc); fa = sum(s["n_fa"] for s in sc)
    return ndet / max(nsz, 1), fa / max(h, 1e-9)


def select(caches, idx, target, sigs):
    best = None
    for sig in sigs:
        c = caches[sig]
        for rr in R_GRID:
            for D in D_GRID:
                sens, fah = agg(score_set(c, idx, np.log(rr), D))
                if fah <= target and (best is None or sens > best[0] or (sens == best[0] and fah < best[1])):
                    best = (sens, fah, sig, rr, D)
    return best


def cv(caches, rows, folds, target, sigs):
    """folds: list of held-out index lists.  returns per-record scored list + choices."""
    allidx = list(range(len(rows)))
    held, choices = [], []
    for te in folds:
        tr = [i for i in allidx if i not in set(te)]
        b = select(caches, tr, target, sigs)
        choices.append(dict(sig=b[2], r=b[3], D=b[4], train_sens=b[0], train_fah=b[1],
                            held_out=sorted({subject(rows[i]["case"]) for i in te})))
        held += score_set(caches[b[2]], te, np.log(b[3]), b[4])
    return held, choices


# ── summaries ────────────────────────────────────────────────────────────────
def summarise(sc, rows, B=2000, seed=0):
    rng = np.random.default_rng(seed)
    subj = sorted({s["subject"] for s in sc})
    per = {u: dict(n_sz=0, n_det=0, n_fa=0, hours=0.0, ictal_h=0.0, lat=[]) for u in subj}
    for s in sc:
        p = per[s["subject"]]
        p["n_sz"] += s["n_sz"]; p["n_det"] += sum(s["detected"]); p["n_fa"] += s["n_fa"]; p["hours"] += s["hours"]
        p["lat"] += [l for l in s["latency"] if l == l]
        r = rows[s["i"]] if "i" in s else None
        if r is not None:
            p["ictal_h"] += sum(off - on for on, off in r["seizures"]) / 3600
    nsz = sum(p["n_sz"] for p in per.values()); ndet = sum(p["n_det"] for p in per.values())
    fa = sum(p["n_fa"] for p in per.values()); h = sum(p["hours"] for p in per.values())
    ih = h - sum(p["ictal_h"] for p in per.values())
    macro = [p["n_det"] / p["n_sz"] for p in per.values() if p["n_sz"]]
    lats = [l for p in per.values() for l in p["lat"]]
    a = np.array([[per[u]["n_det"], per[u]["n_sz"], per[u]["n_fa"], per[u]["hours"]] for u in subj], float)
    ix = rng.integers(0, len(subj), (B, len(subj)))
    bs = a[ix].sum(1)
    sens_b = bs[:, 0] / np.maximum(bs[:, 1], 1); fah_b = bs[:, 2] / bs[:, 3]
    mac_b = []
    for row in ix:
        v = [a[j, 0] / a[j, 1] for j in row if a[j, 1] > 0]
        mac_b.append(np.mean(v))
    return dict(seizures=int(nsz), detected=int(ndet), sensitivity=ndet / max(nsz, 1),
                sens_ci=[float(np.percentile(sens_b, 2.5)), float(np.percentile(sens_b, 97.5))],
                macro_sensitivity=float(np.mean(macro)),
                macro_ci=[float(np.percentile(mac_b, 2.5)), float(np.percentile(mac_b, 97.5))],
                false_alarms=int(fa), hours=h, fa_per_hour=fa / h, fa_per_hour_interictal=fa / ih,
                fah_ci=[float(np.percentile(fah_b, 2.5)), float(np.percentile(fah_b, 97.5))],
                latency_median=float(np.median(lats)) if lats else float("nan"),
                latency_iqr=[float(np.percentile(lats, 25)), float(np.percentile(lats, 75))] if lats else [float("nan")] * 2,
                subjects=len(subj),
                per_subject={u: dict(n_sz=per[u]["n_sz"], n_det=per[u]["n_det"], n_fa=per[u]["n_fa"], hours=per[u]["hours"]) for u in subj})


def null_test(sc, rows, n_perm, seed=1):
    """circular-shift empirical null for the sensitivity of a set of scored records."""
    rng = np.random.default_rng(seed)
    obs = sum(sum(s["detected"]) for s in sc)
    tot = sum(s["n_sz"] for s in sc)
    null = np.zeros(n_perm)
    for k in range(n_perm):
        det = 0
        for s in sc:
            r = rows[s["i"]]
            if not r["seizures"]:
                continue
            dur = r["duration_s"]
            al = np.asarray(s["alarms"], float)
            if len(al) == 0:
                continue
            sh = (al + rng.uniform(0, dur)) % dur
            for on, off in r["seizures"]:
                det += bool(np.any((sh >= on - PRE_S) & (sh <= off + POST_S)))
        null[k] = det
    return dict(observed=obs / tot, null_mean=float(null.mean() / tot), null_p95=float(np.percentile(null, 95) / tot),
                p=float((1 + np.sum(null >= obs)) / (1 + n_perm)))


def shipped_scored(rows, recs, det, kind):
    sc = []
    for i, r in enumerate(rows):
        al = recs[r["record"]][det][kind]
        s = score_record(al, [tuple(x) for x in r["seizures"]], r["duration_s"])
        s["subject"] = subject(r["case"]); s["alarms"] = al; s["i"] = i
        sc.append(s)
    return sc


def wilcoxon_subjects(a, b):
    from scipy.stats import wilcoxon
    u = sorted(set(a["per_subject"]) & set(b["per_subject"]))
    x = np.array([a["per_subject"][k]["n_det"] / a["per_subject"][k]["n_sz"] for k in u if a["per_subject"][k]["n_sz"]])
    y = np.array([b["per_subject"][k]["n_det"] / b["per_subject"][k]["n_sz"] for k in u if b["per_subject"][k]["n_sz"]])
    d = x - y
    p = float(wilcoxon(x, y).pvalue) if np.any(d != 0) else 1.0
    return dict(n=int(len(x)), better=int((d > 0).sum()), worse=int((d < 0).sum()), ties=int((d == 0).sum()), p=p)


def main():
    global R_GRID
    ap = argparse.ArgumentParser()
    ap.add_argument("--results", default="results")
    ap.add_argument("--channel", default="FP1-FP2")
    ap.add_argument("--perm", type=int, default=1000)
    a = ap.parse_args()
    res = Path(a.results); ch = a.channel
    allrec = [json.loads(l) for l in (res / "records.jsonl").read_text().splitlines() if l.strip()]
    recs = {r["record"]: r for r in allrec if r["channel"] == ch and not r.get("missing")}
    rows = sorted(recs.values(), key=lambda r: r["record"])
    for r in rows:
        n = int(np.floor(r["duration_s"]))
        r["ll1"], r["dev1"] = series(res, r["record"], ch, n)
        sl, sd = baseline_score(r["ll1"]), baseline_score(r["dev1"])
        r["S"] = {"LL": sl, "DEV": sd, "FUSE": 0.5 * (sl + sd)}
    caches = {k: Cache(rows, k) for k in ("LL", "DEV", "FUSE")}
    subjects = sorted({subject(r["case"]) for r in rows})
    loso = [[i for i, r in enumerate(rows) if subject(r["case"]) == u] for u in subjects]
    two = [[i for i, r in enumerate(rows) if P.fold_of(r["case"]) == f] for f in (0, 1)]
    out = dict(channel=ch, subjects=len(subjects), recordings=len(rows),
               seizures=sum(len(r["seizures"]) for r in rows), hours=sum(r["duration_s"] for r in rows) / 3600,
               cold_start_seizures=sum(1 for r in rows for on, off in r["seizures"] if on < 335),
               grid=dict(r=R_GRID, D=D_GRID), methods={}, choices={}, nulls={}, tests={}, ablation={})
    # shipped detectors
    ship = {}
    for name, det, kind in (("py_blackout", "py", "seizure_alarms"), ("py_any", "py", "any_alarms"),
                            ("fw_blackout", "fw", "seizure_alarms")):
        sc = shipped_scored(rows, recs, det, kind)
        ship[name] = sc
        out["methods"][name] = summarise(sc, rows)
        out["nulls"][name] = null_test(sc, rows, a.perm)
    # BR detectors: LOSO and 2-fold, per-signal and nested
    held = {}
    for scheme, folds in (("loso", loso), ("2fold", two)):
        for tgt in TARGETS:
            for sigs, lab in ((["DEV"], "DEV"), (["LL"], "LL"), (["FUSE"], "FUSE"), (["LL", "DEV", "FUSE"], "NESTED")):
                if scheme == "2fold" and lab == "FUSE":
                    continue
                sc, ch_ = cv(caches, rows, folds, tgt, sigs)
                key = f"{scheme}_{lab}_{tgt}"
                held[key] = sc
                out["methods"][key] = summarise(sc, rows)
                out["choices"][key] = ch_
                print(key, f"{100*out['methods'][key]['sensitivity']:.1f}% @ {out['methods'][key]['fa_per_hour']:.2f}", flush=True)
    for key in ("loso_DEV_1.0", "loso_DEV_0.5", "loso_NESTED_1.0", "loso_NESTED_0.5", "loso_LL_1.0"):
        out["nulls"][key] = null_test(held[key], rows, a.perm)
    M = out["methods"]
    out["tests"]["loso_DEV_0.5_vs_py_blackout"] = wilcoxon_subjects(M["loso_DEV_0.5"], M["py_blackout"])
    out["tests"]["loso_DEV_1.0_vs_py_any"] = wilcoxon_subjects(M["loso_DEV_1.0"], M["py_any"])
    out["tests"]["loso_DEV_1.0_vs_LL_1.0"] = wilcoxon_subjects(M["loso_DEV_1.0"], M["loso_LL_1.0"])
    out["tests"]["loso_NESTED_0.5_vs_py_blackout"] = wilcoxon_subjects(M["loso_NESTED_0.5"], M["py_blackout"])
    # excluding cold-start seizures
    for key in ("loso_DEV_1.0", "loso_DEV_0.5"):
        sc = held[key]; nsz = ndet = 0
        for s in sc:
            r = rows[s["i"]]
            for (on, off), d in zip(r["seizures"], s["detected"]):
                if on >= 335:
                    nsz += 1; ndet += d
        out["methods"][key]["sensitivity_excl_coldstart"] = ndet / nsz
        out["methods"][key]["seizures_excl_coldstart"] = nsz
    # ablations (LOSO, DEV, targets 1.0 and 0.5)
    variants = {"lag0": dict(lag=0), "lag10": dict(lag=10), "lag60": dict(lag=60),
                "win60": dict(win=60), "win120": dict(win=120), "win600": dict(win=600),
                "mean": dict(stat="mean")}
    for vname, kw in variants.items():
        for r in rows:
            r["S"]["ABL"] = baseline_score(r["dev1"], **kw)
        c = {"ABL": Cache(rows, "ABL")}
        out["ablation"][vname] = {}
        for tgt in (1.0, 0.5):
            sc, chs = cv(c, rows, loso, tgt, ["ABL"])
            s = summarise(sc, rows, B=200)
            out["ablation"][vname][str(tgt)] = {k: s[k] for k in ("sensitivity", "macro_sensitivity", "fa_per_hour", "latency_median")}
            print("ablation", vname, tgt, f"{100*s['sensitivity']:.1f}% @ {s['fa_per_hour']:.2f}", flush=True)
    # un-normalized absolute threshold on per-second deviation (ADC units)
    for r in rows:
        x = r["dev1"].astype(float).copy(); x[:330] = np.nan
        r["S"]["ABS"] = np.log(np.maximum(x, 1e-9))
    c = {"ABS": Cache(rows, "ABS")}
    saved = R_GRID
    R_GRID = [float(v) for v in np.round(np.geomspace(5, 400, 24), 2)]   # absolute ADC thresholds
    out["ablation"]["absolute"] = {}
    for tgt in (1.0, 0.5):
        sc, chs = cv(c, rows, loso, tgt, ["ABS"])
        s = summarise(sc, rows, B=200)
        out["ablation"]["absolute"][str(tgt)] = {k: s[k] for k in ("sensitivity", "macro_sensitivity", "fa_per_hour", "latency_median")}
        print("ablation absolute", tgt, f"{100*s['sensitivity']:.1f}% @ {s['fa_per_hour']:.2f}", flush=True)
    R_GRID = saved
    (res / f"rigor_{ch}.json").write_text(json.dumps(out, indent=1, default=float))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile {WORK}/evaluation/hw_robustness.py
"""
Hardware-robustness addendum for DEV-BR (FP1-FP2).

Re-runs only the (fast) firmware stage under several front-end conditions and
evaluates DEV-BR on each, so that the paper's gain-invariance and
hardware-sensitivity statements rest on measurements rather than argument.

Conditions (all with the 0.5–40 Hz pass-band model):
  g500, g1000, g2000      amplifier gain ×500 / ×1000 / ×2000, ideal 3.3-V ADC
  g1000_clip, g2000_clip  LM358 ceiling: output cannot exceed 1.8 V (VCC − 1.5 V)
  g1000_adc31             ESP32 default 11-dB attenuation: ≈3.1-V full scale
  g1000_166Hz             loop slower than nominal: 6 ms per sample (≈166.7 Hz)

For every condition:
  • DEV-BR re-tuned by leave-one-subject-out CV (targets 1.0 and 0.5 FA/h)
  • DEV-BR with FIXED parameters (those chosen at g1000) — the invariance test
  • the shipped firmware detector's BLACKOUT alarms, for reference
  • ADC clipping fraction and median per-second deviation

usage: python hw_robustness.py --data data/chbmit --out results_hw
"""
from __future__ import annotations

import argparse
import json
import sys
import time
from multiprocessing import Pool
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
from chbmit import discover, load_record, to_device_adc                  # noqa: E402
from detectors import run_firmware_detector                              # noqa: E402
from scoring import score_record, alarm_times                            # noqa: E402
import personalize as P                                                  # noqa: E402
import rigor as RG                                                       # noqa: E402

CH = "FP1-FP2"
COND = {
    "g500": dict(gain=500.0), "g1000": dict(gain=1000.0), "g2000": dict(gain=2000.0),
    "g1000_clip": dict(gain=1000.0, clip_hi_v=1.8), "g2000_clip": dict(gain=2000.0, clip_hi_v=1.8),
    "g1000_adc31": dict(gain=1000.0, vref=3.1), "g1000_166Hz": dict(gain=1000.0, fs_out=1000.0 / 6.0),
}


def one(job):
    case, name, seizures, data_dir = job
    rec = load_record(Path(data_dir), case, name, seizures)
    if CH not in rec.uv:
        return None
    out = dict(case=case, record=name, seizures=seizures, duration_s=rec.duration_s, cond={})
    for cname, kw in COND.items():
        fs = kw.get("fs_out", 200.0)
        adc = to_device_adc(rec.uv[CH], rec.fs, **kw)
        ncal, t, st, trace = run_firmware_detector(adc, SAMPLE_MS=1000.0 / fs)
        d = trace[1].astype(np.float64)
        n = int(np.floor(rec.duration_s))
        sec = np.clip(((np.arange(len(d)) + ncal) / fs).astype(np.int64), 0, n - 1)
        s = np.bincount(sec, weights=d, minlength=n); c = np.bincount(sec, minlength=n)
        delta = np.where(c > 0, s / np.maximum(c, 1), np.nan)
        idx = np.where(np.isfinite(delta), np.arange(n), 0); np.maximum.accumulate(idx, out=idx)
        delta = delta[idx]
        out["cond"][cname] = dict(delta=delta.astype(np.float32).tolist(),
                                  fw_alarms=alarm_times(t, st, "seizure"),
                                  sat=float(np.mean((adc == 0) | (adc == 4095) |
                                                    ((adc >= np.round(kw["clip_hi_v"] / (kw.get("vref", 3.3) / 4096))) if kw.get("clip_hi_v") else False))))
    return out


def evaluate(recs, outdir):
    res = {}
    subjects = sorted({RG.subject(r["case"]) for r in recs})
    loso = [[i for i, r in enumerate(recs) if RG.subject(r["case"]) == u] for u in subjects]
    fixed = None
    for cname in COND:
        rows = []
        for r in recs:
            d = np.asarray(r["cond"][cname]["delta"], float)
            rows.append(dict(case=r["case"], record=r["record"], seizures=r["seizures"], duration_s=r["duration_s"],
                             S={"DEV": RG.baseline_score(d)}))
        cache = {"DEV": RG.Cache(rows, "DEV")}
        e = dict(sat=float(np.mean([r["cond"][cname]["sat"] for r in recs])),
                 median_delta=float(np.nanmedian(np.concatenate([np.asarray(r["cond"][cname]["delta"], float) for r in recs]))))
        for tgt in (1.0, 0.5):
            sc, chs = RG.cv(cache, rows, loso, tgt, ["DEV"])
            s = RG.summarise(sc, rows, B=500)
            e[f"loso_{tgt}"] = {k: s[k] for k in ("sensitivity", "sens_ci", "macro_sensitivity", "fa_per_hour", "latency_median")}
            e[f"loso_{tgt}"]["choices"] = [(c["r"], c["D"]) for c in chs]
        if cname == "g1000":
            # the single most common (r, D) chosen across LOSO folds at each target → frozen parameters
            from collections import Counter
            fixed = {}
            for tgt in (1.0, 0.5):
                fixed[tgt] = Counter(map(tuple, e[f"loso_{tgt}"]["choices"])).most_common(1)[0][0]
        res[cname] = e
        res[cname]["rows"] = rows
    for cname in COND:
        rows = res[cname].pop("rows")
        cache = RG.Cache(rows, "DEV")
        for tgt, (rr, D) in fixed.items():
            sc = RG.score_set(cache, list(range(len(rows))), np.log(rr), D)
            s = RG.summarise(sc, rows, B=500)
            res[cname][f"fixed_{tgt}"] = dict(r=rr, D=D, **{k: s[k] for k in ("sensitivity", "sens_ci", "fa_per_hour", "latency_median")})
        fw = []
        for r in recs:
            x = score_record(r["cond"][cname]["fw_alarms"], [tuple(v) for v in r["seizures"]], r["duration_s"])
            x["case"] = r["case"]; fw.append(x)
        from scoring import summarise as S0
        f = S0(fw)
        res[cname]["firmware_blackout"] = dict(sensitivity=f["sensitivity"], fa_per_hour=f["fa_per_hour"])
        print(cname, json.dumps({k: (v if not isinstance(v, dict) else {kk: vv for kk, vv in v.items() if kk in ('sensitivity', 'fa_per_hour')}) for k, v in res[cname].items()}), flush=True)
    res["fixed_params"] = {str(k): v for k, v in fixed.items()}
    return res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True)
    ap.add_argument("--out", default="results_hw")
    ap.add_argument("--jobs", type=int, default=2)
    a = ap.parse_args()
    out = Path(a.out); out.mkdir(parents=True, exist_ok=True)
    jobs = list(discover(Path(a.data)))
    t0 = time.time()
    with Pool(a.jobs) as pool:
        recs = [r for r in pool.imap_unordered(one, jobs) if r is not None]
    recs.sort(key=lambda r: r["record"])
    print(f"{len(recs)} recordings processed in {time.time()-t0:.0f}s", flush=True)
    np.savez_compressed(out / "delta_series.npz",
                        **{f"{r['record']}__{c}": np.asarray(r['cond'][c]['delta'], np.float32) for r in recs for c in COND})
    res = evaluate(recs, out)
    res["recordings"] = len(recs)
    res["seizures"] = sum(len(r["seizures"]) for r in recs)
    (out / "hw_robustness.json").write_text(json.dumps(res, indent=1, default=float))


if __name__ == "__main__":
    main()


In [ ]:
t0 = time.time()
if not os.environ.get("VERA_DATA"):
    sh(f"{sys.executable} {WORK}/evaluation/fetch_chbmit.py --out {DATA} --workers 6")
print(len(list(Path(DATA).glob('*.ch.json'))), "recordings ready", f"{time.time()-t0:.0f}s")

In [ ]:
t0 = time.time()
sh(f"{sys.executable} {WORK}/evaluation/hw_robustness.py --data {DATA} --out {OUT} --jobs {JOBS}")
print(f"done in {(time.time()-t0)/60:.1f} min")

In [ ]:
import platform, numpy, scipy, numba
json.dump(dict(python=platform.python_version(), numpy=numpy.__version__, scipy=scipy.__version__, numba=numba.__version__,
               finished=time.strftime("%Y-%m-%d %H:%M:%S")), open(f"{OUT}/manifest.json", "w"), indent=2)
shutil.copytree(f"{WORK}/evaluation", f"{OUT}/code", dirs_exist_ok=True)
shutil.make_archive(f"{WORK}/vera_hw_results", "zip", OUT)
print("vera_hw_results.zip", os.path.getsize(f"{WORK}/vera_hw_results.zip") // 2**10, "KB")
if os.path.isdir("/kaggle/working"): shutil.rmtree(DATA, ignore_errors=True)
R = json.load(open(f"{OUT}/hw_robustness.json"))
for c, v in R.items():
    if isinstance(v, dict) and "loso_1.0" in v:
        print(f"{c:12s} LOSO≤1: {100*v['loso_1.0']['sensitivity']:.1f}% @ {v['loso_1.0']['fa_per_hour']:.2f} | "
              f"fixed: {100*v['fixed_1.0']['sensitivity']:.1f}% @ {v['fixed_1.0']['fa_per_hour']:.2f} | sat {100*v['sat']:.2f}%")